# HydroMet — Run Guide

**Collection ID:** `EO.DWD.STAT.HYDROMET_EXTREMES`
**Owner:** Carl Hartick (DWD)

Catalog of extreme precipitation events from DestinE Climate DT simulations,
produced by the HydroMet application of the Climate DT workflow. Combines DWD's
KOSTRA (depth-duration-frequency statistics) and CatRaRE (radar-based event
catalogue) products.

The workflow already writes the STAC layout (with `HYDROMET.UPLOAD_READY: true`),
so this repo takes its output **as it is** — folder names, periods and the
`__model__…__resolution` suffix all come from the workflow. This notebook runs
the whole pipeline on small fake workflow output (or on your own, see step 0) in
a throwaway sandbox and stops **before** the upload step. No credentials needed,
nothing is pushed to S3.

For the conda environment and `.env` credential setup, see the top-level
[`README.md`](../../README.md). For how HydroMet compares to the other three apps,
see [`apps/RUN_GUIDE.md`](../RUN_GUIDE.md).

Prefer a plain shell script over a notebook? See [`run_example.sh`](run_example.sh)
for the same steps with no narration.

## Files in this app

| File | Purpose |
|---|---|
| `catalog_config.yaml` | ALL HydroMet input: collection metadata, `collection_root_dir`, paths, datacube extension, two scientific publications |
| `staging/workflow_items.py` | The workflow's item folder naming scheme, shared by `stage.py` and `move.py` |
| `staging/check_data.sh` | Optional pre-staging QC of workflow output: item names, year folders, KO_params/DDF files, event CSVs |
| `staging/stage.py` | Writes `collection.json` + `collection_config.json`; lists the workflow's item folders and flags invalid names. Creates **no** item folders |
| `staging/move.py` | Gathers the item folders of one or more workflow outputs into one collection root, names unchanged |
| `staging/update.py` | Generates STAC item metadata for the collection |
| `staging/delete_s3_collection_prefix.py` | Maintenance utility to delete a collection prefix from the S3 bucket |
| `run_pipeline.sh` | Wraps `stage.py` + `update.py` + upload into one command (does **not** run `move.py`) |

### What the workflow writes

```
<HPCROOTDIR>/output/hydromet/<DATELIST>/<memberNN>/
└── EO.DWD.STAT.HYDROMET/                         ← bucket root (collection_root_dir)
    └── data/1990/                                ← start year of the experiment
        └── EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/
            ├── 1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc
            ├── 1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc
            └── 1990_01_01_T00_00_to_2014_12_31_T23_00_events_T5_<REGION>.csv   (one per region)
```

`EO.DWD.STAT.HYDROMET` is not an extra folder level: its `metadata/` and `data/`
land at the root of the bucket. One item per experiment, covering the
experiment's own period (`item_folder_level: YYYY`).

In [1]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import yaml

APP_DIR = Path.cwd()
assert (APP_DIR / "catalog_config.yaml").exists(), "run this notebook from apps/hydromet/"
REPO = APP_DIR.parents[1]

COLLECTION_ID = "EO.DWD.STAT.HYDROMET_EXTREMES"
SANDBOX = Path(tempfile.mkdtemp(prefix="hydromet_notebook_"))
print(f"Sandbox: {SANDBOX}")

def run(*args, home, check=True):
    """Run a staging script with CATALOG_HOME_DIR=home."""
    print("$", " ".join(map(str, args)))
    env = {**os.environ, "CATALOG_HOME_DIR": str(home)}
    result = subprocess.run(args, cwd=APP_DIR, env=env, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="")  # stage.py/move.py/update.py log to stderr
    if check:
        result.check_returncode()

def show_tree(root):
    for p in sorted(root.rglob("*")):
        print("  " * (len(p.relative_to(root).parts) - 1) + p.name + ("/" if p.is_dir() else ""))

Sandbox: /tmp/hydromet_notebook_xxxxxxxx


## 1. The config file

`collection_root_dir` is the folder the workflow writes; `id` stays the collection
id used in item ids. `property_order` is the order of the workflow's `__` suffix.

In [2]:
config = yaml.safe_load((APP_DIR / "catalog_config.yaml").read_text())
print("id:                 ", config["id"])
print("collection_root_dir:", config["collection_root_dir"])
print("license:            ", config["license"])
print("item_folder_level:  ", config["item_folder_level"])
print("property_order:     ", config["property_order"])
print("publications:       ", [p["doi"] for p in config["publications"]])

id:                  EO.DWD.STAT.HYDROMET_EXTREMES
collection_root_dir: EO.DWD.STAT.HYDROMET
license:             CC-BY-4.0
item_folder_level:   YYYY
property_order:      ['model', 'experiment', 'generation', 'realization', 'activity', 'resolution']
publications:        ['10.1127/metz/2021/1088', '10.5194/hess-27-1109-2023']


## 2. Step 0 — the workflow output

Set `MEMBERS` to real workflow member folders
(`<HPCROOTDIR>/output/hydromet/<DATELIST>/<memberNN>`) to use your own data. Left
empty, two fake experiments (historical 1990–2014 and SSP3-7.0 2015–2049) are
generated in the sandbox, laid out exactly like the workflow writes them.

In [3]:
MEMBERS = []  # <-- your workflow member folders

if not MEMBERS:
    subprocess.run([sys.executable, str(REPO / "tools" / "generate_sample_data.py"), "hydromet", str(SANDBOX)],
                   check=True, capture_output=True)
    MEMBERS = [SANDBOX / "appdata" / "hist", SANDBOX / "appdata" / "ssp370"]

for member in MEMBERS:
    print(f"{member}/")
    show_tree(Path(member))

/tmp/hydromet_notebook_xxxxxxxx/appdata/hist/
EO.DWD.STAT.HYDROMET/
  data/
    1990/
      EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/
        1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc
        1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc
        1990_01_01_T00_00_to_2014_12_31_T23_00_events_T5_med.csv
        1990_01_01_T00_00_to_2014_12_31_T23_00_events_T5_neu.csv
/tmp/hydromet_notebook_xxxxxxxx/appdata/ssp370/
EO.DWD.STAT.HYDROMET/
  data/
    2015/
      EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__high/
        2015_01_01_T00_00_to_2049_01_01_T00_00_DDF.nc
        2015_01_01_T00_00_to_2049_01_01_T00_00_KO_params.nc
        2015_01_01_T00_00_to_2049_12_31_T23_00_events_T5_med.csv
        2015_01_01_T00_00_to_2049_12_31_T23_00_events_T5_neu.csv


## 3. `check_data.sh` (optional): QC of the workflow output

Checks the item names, that each item sits in the folder of its start year, that
the KO_params/DDF files are there, and counts the event CSVs.

In [4]:
for member in MEMBERS:
    result = subprocess.run(["bash", "staging/check_data.sh", str(member)],
                            cwd=APP_DIR, capture_output=True, text=True)
    print(result.stdout)

=== Checking /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET/data ===
--- 1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high
period: 19900101 to 20141231
OK: *_KO_params.nc (1)
OK: *_DDF.nc (1)
event CSVs: 2
=== 1 item(s), 0 problem(s) ===

=== Checking /tmp/hydromet_notebook_xxxxxxxx/appdata/ssp370/EO.DWD.STAT.HYDROMET/data ===
--- 2015/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__high
period: 20150101 to 20491231
OK: *_KO_params.nc (1)
OK: *_DDF.nc (1)
event CSVs: 2
=== 1 item(s), 0 problem(s) ===



## 4. Option A — one experiment: build the catalog in place

With `CATALOG_HOME_DIR` set to the member folder, `stage.py` writes `metadata/`
next to the workflow's `data/`, and `update.py` generates the items. No files are
moved or renamed; `move.py` is not needed. This is what `run_pipeline.sh` does
when `CATALOG_HOME_DIR` points at a member folder.

In [5]:
run(sys.executable, "staging/stage.py", "--execute", home=MEMBERS[0])
run(sys.executable, "staging/update.py", home=MEMBERS[0])

$ python staging/stage.py --execute


INFO | Collection: EO.DWD.STAT.HYDROMET_EXTREMES in /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET
INFO | Wrote /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET/metadata/collection.json and /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET/metadata/collection_config.json
INFO | Item 1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high (1990-01-01 to 2014-12-31)
INFO | Workflow items: {'items': 1, 'invalid': 0}
$ python staging/update.py


PySTAC version: 1.0.0
Collection passed all field checks.
item_config_optional:True
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year_from_folder_path:1990, month_from_folder_path:01, day_from_folder_path:01
✔ Sorted assets in: EO.DWD.STAT.HYDROMET/metadata/items/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high.json
INFO | Generating STAC item metadata for EO.DWD.STAT.HYDROMET_EXTREMES
WARNING | Bucket credentials not set, but IS_UPLOAD_S3 is False. This is ok. Skipping upload mode.
INFO | Successfully validated stac collection: EO.DWD.STAT.HYDROMET_EXTREMES
INFO | Optional Config file not found: EO.DWD.STAT.HYDROMET/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/item_config.json. Returning empty dictionary.
INFO | item_folder_path:EO.DWD.STAT.HYDROMET/data/1990/EO.D

## 5. Option B — several experiments: gather them into one catalog

### Step 1 — `stage.py`

Writes `metadata/collection.json` + `metadata/collection_config.json` into
`<CATALOG_HOME_DIR>/EO.DWD.STAT.HYDROMET/`. It creates no item folders: they come
from the workflow.

In [6]:
CATALOG_HOME = SANDBOX / "catalog"
CATALOG_HOME.mkdir()
run(sys.executable, "staging/stage.py", home=CATALOG_HOME)  # dry run

$ python staging/stage.py


INFO | Collection: EO.DWD.STAT.HYDROMET_EXTREMES in /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDROMET
INFO | [DRY RUN] Would write collection.json + collection_config.json
INFO | No data/ folder yet in /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDROMET
INFO | Workflow items: {'items': 0, 'invalid': 0}
INFO | Dry run complete. Re-run with --execute to apply.


In [7]:
run(sys.executable, "staging/stage.py", "--execute", home=CATALOG_HOME)

$ python staging/stage.py --execute


INFO | Collection: EO.DWD.STAT.HYDROMET_EXTREMES in /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDROMET
INFO | Wrote /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDROMET/metadata/collection.json and /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDROMET/metadata/collection_config.json
INFO | Workflow items: {'items': 0, 'invalid': 0}


### Step 2 — `move.py`: gather the workflow's item folders

One `--source-dir` per workflow output (member folder, its `EO.DWD.STAT.HYDROMET`
folder, or `data/`). Year folder, item folder and file names are kept exactly; the
period of each item is the one the workflow wrote. Item folders with a name that
does not follow the scheme are reported and skipped.

In [8]:
move_args = [sys.executable, "staging/move.py"]
for member in MEMBERS:
    move_args += ["--source-dir", str(member)]
move_args.append("--link")
run(*move_args, home=CATALOG_HOME)  # dry run

$ python staging/move.py --source-dir /tmp/hydromet_notebook_xxxxxxxx/appdata/hist --source-dir /tmp/hydromet_notebook_xxxxxxxx/appdata/ssp370 --link


INFO | Item EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high (1990-01-01 to 2014-12-31, model=IFS-NEMO, experiment=hist, generation=2, realization=1, activity=baseline, resolution=high)
INFO | [symlink] /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc -> /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDROMET/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc
INFO | [symlink] /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc -> /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDR

In [9]:
run(*move_args, "--execute", home=CATALOG_HOME)

$ python staging/move.py --source-dir /tmp/hydromet_notebook_xxxxxxxx/appdata/hist --source-dir /tmp/hydromet_notebook_xxxxxxxx/appdata/ssp370 --link --execute


INFO | Item EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high (1990-01-01 to 2014-12-31, model=IFS-NEMO, experiment=hist, generation=2, realization=1, activity=baseline, resolution=high)
INFO | [symlink] /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc -> /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDROMET/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc
INFO | [symlink] /tmp/hydromet_notebook_xxxxxxxx/appdata/hist/EO.DWD.STAT.HYDROMET/data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc -> /tmp/hydromet_notebook_xxxxxxxx/catalog/EO.DWD.STAT.HYDR

### Step 3 — `update.py`: generate STAC item metadata

In [10]:
run(sys.executable, "staging/update.py", home=CATALOG_HOME)

$ python staging/update.py


PySTAC version: 1.0.0
Collection passed all field checks.
item_config_optional:True
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year_from_folder_path:1990, month_from_folder_path:01, day_from_folder_path:01
item_config_optional:True
item_id:EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__high, collection_id:EO.DWD.STAT.HYDROMET_EXTREMES, year_from_folder_path:2015, month_from_folder_path:01, day_from_folder_path:01
✔ Sorted assets in: EO.DWD.STAT.HYDROMET/metadata/items/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high.json
✔ Sorted assets in: EO.DWD.STAT.HYDROMET/metadata/items/EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__high.json
INFO | Generating STAC item metadata for EO.DWD.STAT.HYDROMET_EXTREMES
WARNING | Bucket 

## 6. Inspect the results

In [11]:
items_dir = CATALOG_HOME / config["collection_root_dir"] / "metadata" / "items"
item_paths = sorted(items_dir.glob("*.json"))
print(f"{len(item_paths)} item JSON(s) generated:")
for p in item_paths:
    print(" ", p.name)

2 item JSON(s) generated:
  EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high.json
  EO.DWD.STAT.HYDROMET_EXTREMES_20150101T000000_20491231T230000__IFS-NEMO__SSP3-7.0__2__1__projections__high.json


In [12]:
sample = json.loads(item_paths[0].read_text())
print("id:        ", sample["id"])
print("collection:", sample["collection"])
print("properties:", sample["properties"])
print("assets:    ", {k: v["href"] for k, v in sample["assets"].items()})

id:         EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high
collection: EO.DWD.STAT.HYDROMET_EXTREMES
properties: {'model': 'IFS-NEMO', 'experiment': 'hist', 'generation': '2', 'realization': '1', 'activity': 'baseline', 'resolution': 'high', 'start_datetime': '1990-01-01T00:00:00Z', 'end_datetime': '2014-12-31T23:00:00Z', 'datetime': '1990-01-01T00:00:00Z'}
assets:     {'1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc': 'data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_DDF.nc', '1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc': 'data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hist__2__1__baseline__high/1990_01_01_T00_00_to_2014_01_01_T00_00_KO_params.nc', '1990_01_01_T00_00_to_2014_12_31_T23_00_events_T5_med.csv': 'data/1990/EO.DWD.STAT.HYDROMET_EXTREMES_19900101T000000_20141231T230000__IFS-NEMO__hi

## 7. Upload (not run in this notebook)

Skipped here on purpose. To actually publish:

1. Copy `apps/hydromet/.env.example` → `apps/hydromet/.env` and fill in
   `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` (see the top-level README's
   "Configure credentials" section).
2. Set `CATALOG_HOME_DIR` to the member folder (option A) or the gathered
   catalog (option B).
3. Run:
   ```bash
   bash run_pipeline.sh --execute --upload
   bash run_pipeline.sh --execute --upload --year 1990   # resume/retry a single year
   ```

The contents of `EO.DWD.STAT.HYDROMET/` go to the bucket root (`metadata/…`,
`data/…`).

In [13]:
import shutil
shutil.rmtree(SANDBOX)
print(f"Removed sandbox: {SANDBOX}")

Removed sandbox: /tmp/hydromet_notebook_xxxxxxxx
